In [30]:
# Для работы с html файлами
from bs4 import BeautifulSoup

# import requests as req

# Для работы с таблицами
import pandas as pd

# Для работы со смайликами в тексте
import emoji

# Для превращения смайликов в слова
import demoji

# pip install googletrans==3.1.0a0
# Переводчик
from googletrans import Translator

# Для VK
from vk_api import VkApi

# Telegram (проверка данных, полученных с помощью парсера)

In [33]:
df_telegram = pd.read_csv('chats.csv')

In [34]:
df_telegram.head(10)

,message
0,Транспортная система — основа современной экон...
1,"➡️Что-то модно, а что-то вечно — рассказываем ..."
2,NaN
3,NaN
4,NaN
5,NaN
6,NaN
7,NaN
8,Заместитель Председателя Правительства Виталий...
9,NaN


In [35]:
df_telegram.iloc[1].message

'➡️Что-то модно, а что-то вечно — рассказываем о легендарных приметах Российского университета транспорта, которые смогут прокачать твой день! \r\n\r\nВстречай топ-5 легенд РУТ ⚡️'

# Telegram  xml (чтение файла с сообщениями, полученного через экспорт сообщений группы в Desktop приложении)

In [ ]:
HTMLFile = open("messages.html", "rb")

In [107]:
# Reading the file
index = HTMLFile.read()

In [ ]:
# Creating a BeautifulSoup object and specifying the parser
S = BeautifulSoup(index, 'lxml')
S

In [109]:
item = S.find_all('div', class_='text')
set_text = []
for name in item:
    set_text.append(name.text)

In [110]:
len(set_text)

443

In [112]:
set_text = [line.strip() for line in set_text]

In [ ]:
set_text

In [86]:
df_telegram = pd.DataFrame(set_text, columns = ['text'])
df_telegram['goal'] = 'аэрофлот'
df_telegram[:2]

,text,goal
0,АЭРОФЛОТ,аэрофлот
1,Добро пожаловать на канал Аэрофлота! ✈️,аэрофлот


# Извелчение смайликов из текстов сообщений

In [115]:
def extract_emojis(s):
    return ''.join(c for c in s if c in emoji.EMOJI_DATA)

In [116]:
emoji_chat = [extract_emojis(s) for s in set_text]

In [ ]:
emoji_chat

In [ ]:
# demoji.download_codes()

In [124]:
# Декодирования смайликов
dict_emoj = demoji.findall(emoji_chat[1])

In [125]:
# Создаем словарь
res = []
for key in dict_emoj.keys() :    res.append(dict_emoj[key])

In [127]:
res

['airplane']

In [20]:
# Перевод на русский
translator = Translator()

In [25]:
result = translator.translate('airplane', dest='ru')

In [27]:
result.text

'самолет'

https://dev-gang.ru/article/perevod-teksta-s-pomosczu-google-translate-api-v-python-ahgm88wx1k/

# Парсинг вконтакте


In [3]:
# Вставьте сюда свой access_token, полученный из адресной строки!
token='vk1.a...'

In [4]:
# Функция для парсинга постов со стены выбранной группы ВК через group_id
def main(offset: int, token: str, group_id: str):
    vk = vk_api.VkApi(token=token) # авторизация через токен 
    api = vk.get_api()
    posts = api.wall.get(owner_id = group_id, offset = offset, count=100)['items']
    posts_strings = [post['text'] for post in posts]
    num_like = []
    comments_strings = []
    for post in posts:
        comments = api.wall.getComments(owner_id = group_id, post_id=post['id'], count=100)['items']
        comments_strings.append([comment['text'] for comment in comments])
        itemID = post['id']
        isLiked = api.likes.getList(
        type = 'post', 
        owner_id = group_id, 
        item_id = itemID               
        )
        num_like.append(isLiked['count'])
    return posts_strings, comments_strings, num_like

In [5]:
combo_list_posts = []
combolist_comments = []
combolist_like_count = []
for i in range(0, 301, 100):
    try:
        rzd_posts, comments_strings_rzd, rzd_like_count = main(offset = i, token = token, group_id = '-38981315')
        combo_list_posts.extend(rzd_posts)
        combolist_comments.extend(comments_strings_rzd)
        combolist_like_count.extend(rzd_like_count)
    except:
        print('Постов больше нет на смещении: ', i)

Постов больше нет на смещении:  0
Постов больше нет на смещении:  100


In [6]:
len(combo_list_posts)

200

In [7]:
len(set(combo_list_posts))

136

In [8]:
import pandas as pd

In [14]:
# Добавьте функцию для сохранения данных в csv, иначе потеряете их
my_text_1 = pd.DataFrame(data = combo_list_posts)
my_text_1['goal'] = 1
my_text_1.tail()

,0,goal
195,Новый год по-древнерусски! Едем встречать волш...,1
196,,1
197,"За минуту рассказываем всё, что важно знать о ...",1
198,Открыли новые залы ожидания для маломобильных ...,1
199,,1


In [18]:
combolist_comments[:1]

[['Амур',
  'Тульская обл?',
  'Тульская область, речка Любовка, на заднем плане ответвление с плотиной - речка Маклец, справа чуть видно цирканал ГРЭС. Новомосковское кольцо, перегон Ключёвка - Новомосковская-2.',
  'Новомосковск.',
  'Остров Борнео',
  'Здравствуйте. Я могу получить разъяснение, какие скидки при покупке билетов положены вообще инвалидам 3 группы, студентам также? Видимо никаких? Спросила спокойно у кассира на кассах жд вокзала Хабаровска, мне в ответ высокомерно ответили что они не в курсе вообще льгот. Что это было вообще? Сознательное введение в заблуждение? Почему тогда инвалидам детства раз в год можно бесплатно к месту лечения, а если ты инвалид 3 группы или студент, то нет? Я не ною, Я могу заплатить, но ездию редко и не в курсе какие кому льготы положены. Почему об этом нет информации на стендах в здании жд вокзала? Раньше даже возле туалетов было написано на стендах про льготы, теперь и этого нет....',
  'Как же  я скучаю...',
  'Тульская область , станция Но

In [25]:
# Один из способов превращения двумерного списка в одномерный
comments = [message for post in combolist_comments for message in post]

In [26]:
len(comments)

3254

In [29]:
comments[:1]

['Амур']